# LeetCode 2139 — Minimum Moves to Reach Target Score bằng Python

Link đề: https://leetcode.com/problems/minimum-moves-to-reach-target-score/ · Độ khó: **Medium** · Chủ đề: Math, Greedy

Skeleton LeetCode:

```python
class Solution:
    def minMoves(self, target: int, maxDoubles: int) -> int:
        ...
```

## 1. Mục tiêu học

1. Kỹ thuật **đi ngược** (work backwards): từ `target` về `1`, phép "nhân đôi" thành "chia đôi", phép "+1" thành "−1".
2. Chứng minh tham lam "**chẵn thì chia đôi, lẻ thì trừ 1**" bằng lập luận đổi chỗ / so sánh chi phí.
3. Đi từ **BFS trên trạng thái** (chuẩn vàng) → **quy hoạch động có nhớ** → **tham lam lặp** → **công thức bit**.
4. Nhận ra bẫy **O(target)**: khi hết lượt nhân đôi phải cộng thẳng `target − 1`, không được lặp từng bước (target tới 10⁹).
5. Kiểm chứng tham lam bằng BFS trên mọi `target ≤ 200`, mọi `maxDoubles ≤ 8`.

## 2. Đề bài & ví dụ

Bắt đầu với số nguyên `1`, muốn đạt `target`. Mỗi **bước** chọn một trong hai:
- **Tăng 1**: `x = x + 1` (dùng bao nhiêu lần cũng được);
- **Nhân đôi**: `x = 2x` (dùng **tối đa `maxDoubles`** lần).

Trả về **số bước ít nhất** để từ 1 đạt đúng `target`.

**Ví dụ 1:** `target = 5, maxDoubles = 0` → `4` (chỉ tăng: 1→2→3→4→5).

**Ví dụ 2:** `target = 19, maxDoubles = 2` → `7`: 1→2→3→4 (3 lần tăng), ×2 → 8, +1 → 9, ×2 → 18, +1 → 19.

**Ví dụ 3:** `target = 10, maxDoubles = 4` → `4`: 1→2, ×2 → 4, +1 → 5, ×2 → 10.

**Ràng buộc:** `1 ≤ target ≤ 10⁹`, `0 ≤ maxDoubles ≤ 100`.

In [1]:
from functools import lru_cache
from collections import deque
import random, time, sys

examples = [(5, 0, 4), (19, 2, 7), (10, 4, 4)]
def apply(path):
    x = 1
    for op in path:
        x = x + 1 if op == "+1" else 2 * x
    return x
print(apply(["+1"] * 3 + ["x2", "+1", "x2", "+1"]), "sau", 7, "bước")
print(apply(["+1", "x2", "+1", "x2"]), "sau", 4, "bước")

19 sau 7 bước
10 sau 4 bước


## 3. Kiến thức nền

### 3.1 Phép chia lấy nguyên và kiểm tra chẵn lẻ

`t // 2` chia lấy phần nguyên; `t % 2` (hay `t & 1`) cho biết `t` lẻ; `t >> 1` tương đương `t // 2` với số không âm.

In [2]:
for t in [19, 18, 9, 8]:
    print(f"t={t}: t//2={t // 2}, t%2={t % 2}, t&1={t & 1}, t>>1={t >> 1}")

t=19: t//2=9, t%2=1, t&1=1, t>>1=9
t=18: t//2=9, t%2=0, t&1=0, t>>1=9
t=9: t//2=4, t%2=1, t&1=1, t>>1=4
t=8: t//2=4, t%2=0, t&1=0, t>>1=4


### 3.2 BFS tìm đường ngắn nhất trên đồ thị trạng thái

Mỗi trạng thái là `(x, số lần đã nhân đôi)`; mỗi bước là một cạnh trọng số 1 ⇒ BFS cho số bước ít nhất. Chỉ khả thi khi `target` nhỏ.

In [3]:
q = deque([(1, 0)]); q.append((2, 1)); print(q.popleft(), list(q))

(1, 0) [(2, 1)]


### 3.3 `functools.lru_cache` — ghi nhớ kết quả hàm đệ quy

Trang trí `@lru_cache(None)` lên hàm thuần (cùng đầu vào → cùng đầu ra) để không tính lại. Lưu ý giới hạn đệ quy (mặc định 1000).

In [4]:
@lru_cache(None)
def fib(n):
    return n if n < 2 else fib(n - 1) + fib(n - 2)
print(fib(80), fib.cache_info().hits, "lần dùng lại kết quả")
print("giới hạn đệ quy:", sys.getrecursionlimit())

23416728348467685 78 lần dùng lại kết quả
giới hạn đệ quy: 1000


### 3.4 `bin()` và `int.bit_length()`

`bin(19) = '0b10011'`. Đi ngược từ `target`: mỗi lần chia đôi là **bỏ bit cuối**; mỗi lần trừ 1 khi lẻ là **xóa bit 1 cuối** — số bước liên quan trực tiếp tới biểu diễn nhị phân.

In [5]:
for t in [19, 10, 1024, 1023]:
    print(t, bin(t), "bit_length =", t.bit_length(), "số bit 1 =", bin(t).count("1"))

19 0b10011 bit_length = 5 số bit 1 = 3
10 0b1010 bit_length = 4 số bit 1 = 2
1024 0b10000000000 bit_length = 11 số bit 1 = 1
1023 0b1111111111 bit_length = 10 số bit 1 = 10


## 4. Ý tưởng nền tảng / trực giác

**Đi xuôi khó**: từ 1, nên nhân đôi lúc nào? Nhân đôi sớm (khi x nhỏ) gần như vô ích (`1 → 2` chỉ bằng một lần +1). Nhân đôi **càng muộn** (x càng lớn) càng lợi — nhưng "muộn" bao nhiêu thì khó thấy.

**Đi ngược dễ hơn**: từ `target` về 1, phép ngược của "nhân đôi" là "chia đôi" (chỉ làm được khi số **chẵn**), phép ngược của "+1" là "−1". Chia đôi một số **lớn** tiết kiệm rất nhiều bước −1 ⇒ nên chia đôi **sớm** (khi còn lớn) — tức là chia đôi ngay khi có thể.

Tính tay ví dụ 2 ngược từ 19, `maxDoubles = 2`:

| t | chẵn? | còn lượt chia | hành động | bước |
|---|---|---|---|---|
| 19 | lẻ | 2 | −1 → 18 | 1 |
| 18 | chẵn | 2 | ÷2 → 9 | 2 |
| 9 | lẻ | 1 | −1 → 8 | 3 |
| 8 | chẵn | 1 | ÷2 → 4 | 4 |
| 4 | — | 0 | hết lượt: −1 ba lần → 1 | 4 + 3 = 7 |

Đáp án 7. ✔

In [6]:
t, d, moves = 19, 2, 0
while t > 1 and d > 0:
    if t % 2:
        t -= 1; act = "lẻ  -> trừ 1"
    else:
        t //= 2; d -= 1; act = "chẵn -> chia 2"
    moves += 1
    print(f"{act:<16} t={t:<3} còn {d} lượt, moves={moves}")
print(f"hết lượt/hoặc t=1: cộng {t - 1} bước trừ 1 -> tổng {moves + t - 1}")

lẻ  -> trừ 1     t=18  còn 2 lượt, moves=1
chẵn -> chia 2   t=9   còn 1 lượt, moves=2
lẻ  -> trừ 1     t=8   còn 1 lượt, moves=3
chẵn -> chia 2   t=4   còn 0 lượt, moves=4
hết lượt/hoặc t=1: cộng 3 bước trừ 1 -> tổng 7


## 5. Cách 1 — Brute Force / baseline: BFS trên trạng thái `(x, đã nhân đôi)`

**Ý tưởng:** đúng định nghĩa: từ `(1, 0)` đi tới `(x+1, k)` hoặc `(2x, k+1)` nếu `k < maxDoubles`; bỏ trạng thái có `x > target` (không bao giờ quay về được vì các phép chỉ làm tăng).

In [7]:
def brute_bfs(target, maxDoubles):
    start = (1, 0)
    dist = {start: 0}
    q = deque([start])
    while q:
        x, k = q.popleft()
        if x == target:
            return dist[(x, k)]
        nxt = [(x + 1, k)]
        if k < maxDoubles:
            nxt.append((2 * x, k + 1))
        for s in nxt:
            if s[0] <= target and s not in dist:
                dist[s] = dist[(x, k)] + 1
                q.append(s)
    return -1

for t, d, exp in examples:
    got = brute_bfs(t, d)
    print(got, "OK" if got == exp else "SAI")

4 OK
7 OK
4 OK


**Độ phức tạp:** O(target · maxDoubles) trạng thái ⇒ với `target = 10⁹` là không thể.

**Nút thắt:** BFS xét **mọi** cách xen kẽ hai phép. Ta cần hiểu cấu trúc của lời giải tối ưu để khỏi thử hết.

### 5.1 Baseline thứ hai: quy hoạch động ngược có nhớ

`f(t, d)` = số bước ít nhất để từ 1 tới `t` với tối đa `d` lần nhân đôi (đi ngược: từ `t` về 1).
- `t == 1` ⇒ 0. `d == 0` ⇒ `t − 1`.
- Ngược lại: `f(t, d) = min(1 + f(t − 1, d), 1 + f(t/2, d − 1) nếu t chẵn)`.

Không cần nhánh "trừ 1 khi chẵn" quá sâu nhờ ghi nhớ? Vẫn có thể sâu `O(t)` — nên chỉ dùng với `t` nhỏ, và ta tăng tạm giới hạn đệ quy một cách có kiểm soát.

In [8]:
def dp_memo(target, maxDoubles):
    @lru_cache(None)
    def f(t, d):
        if t == 1: return 0
        if d == 0: return t - 1
        best = 1 + f(t - 1, d)
        if t % 2 == 0:
            best = min(best, 1 + f(t // 2, d - 1))
        return best
    return f(target, maxDoubles)

for t, d, exp in examples:
    got = dp_memo(t, d)
    print(got, "OK" if got == exp else "SAI")

4 OK
7 OK
4 OK


Độ sâu đệ quy của `dp_memo` có thể tới ~`target` (chuỗi `t → t−1 → t−2 …`), nên chỉ dùng cho `target ≤ vài trăm`.

## 6. Ý tưởng thuật toán / bước đột phá

**Quy tắc tham lam (đi ngược):**
- **Nếu** `t == 1` **thì** dừng.
- **Nếu** hết lượt chia (`d == 0`) **thì** cộng thẳng `t − 1` và dừng (không lặp từng bước!).
- **Nếu** `t` lẻ **thì** bắt buộc trừ 1 (không chia đôi được số lẻ).
- **Nếu** `t` chẵn và còn lượt **thì** chia đôi.

**Vì sao "chẵn thì chia ngay" là tối ưu?** Giả sử ở `t` chẵn (lớn hơn 2), một lời giải tối ưu lại trừ 1 hai lần trở lên rồi mới chia (hoặc không chia nữa). So sánh hai cách đi từ `t` xuống `t/2 − 1` hoặc thấp hơn:
- Trừ `2j` lần rồi chia: `t → t − 2j → (t − 2j)/2 = t/2 − j`: tốn `2j + 1` bước.
- Chia trước rồi trừ `j` lần: `t → t/2 → t/2 − j`: tốn `1 + j` bước — **ít hơn** (khi `j ≥ 1`), dùng cùng số lượt chia.

(Trừ một số **lẻ** lần rồi chia thì không chia được vì số thành lẻ.) Nếu lời giải tối ưu không chia nữa thì ta chia ngay cũng không tệ hơn vì `t/2 − 1 + 1 ≤ t − 1` khi `t ≥ 2`. Vậy luôn có lời giải tối ưu chia đôi ngay khi `t` chẵn. ∎

**Số vòng lặp:** mỗi 2 vòng tối đa dùng 1 lượt chia ⇒ ≤ `2·maxDoubles` vòng, **không phụ thuộc** `target`.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Tham lam ngược, đệ quy (kiểu doocs)

In [9]:
def greedy_recursive(target, maxDoubles):
    if target == 1:
        return 0
    if maxDoubles == 0:
        return target - 1
    if target % 2 == 0:
        return 1 + greedy_recursive(target // 2, maxDoubles - 1)
    return 1 + greedy_recursive(target - 1, maxDoubles)

for t, d, exp in examples:
    got = greedy_recursive(t, d)
    print(got, "OK" if got == exp else "SAI")

4 OK
7 OK
4 OK


Độ sâu đệ quy ≤ `2·maxDoubles + 1 ≤ 201` — an toàn với giới hạn 1000.

**Độ phức tạp cách 2:** O(min(maxDoubles, log target)) thời gian, cùng cỡ cho ngăn xếp đệ quy.

### 7.2 Cách 3 — Tham lam ngược, vòng lặp

In [10]:
def greedy_iterative(target, maxDoubles):
    moves = 0
    while target > 1 and maxDoubles > 0:
        if target % 2 == 1:
            target -= 1
        else:
            target //= 2
            maxDoubles -= 1
        moves += 1
    return moves + (target - 1)

for t, d, exp in examples:
    got = greedy_iterative(t, d)
    print(got, "OK" if got == exp else "SAI")

4 OK
7 OK
4 OK


In [11]:
class SolutionIter:
    def minMoves(self, target: int, maxDoubles: int) -> int:
        ans = 0
        while maxDoubles and target > 1:
            ans += 1
            if target & 1:
                target -= 1
            else:
                target >>= 1
                maxDoubles -= 1
        return ans + target - 1

print([SolutionIter().minMoves(t, d) for t, d, _ in examples])

[4, 7, 4]


### 7.3 Cách 4 — Gộp "trừ 1 rồi chia" thành một bước bit

Với số lẻ, "−1 rồi ÷2" = `t >> 1` và tốn 2 bước; với số chẵn, "÷2" = `t >> 1` và tốn 1 bước. Vậy mỗi lượt chia: `moves += 1 + (t & 1)`, `t >>= 1`. Dừng khi hết lượt hoặc `t == 1`.

In [12]:
def greedy_bits(target, maxDoubles):
    moves = 0
    while maxDoubles > 0 and target > 1:
        moves += 1 + (target & 1)
        target >>= 1
        maxDoubles -= 1
    return moves + target - 1

for t, d, exp in examples:
    got = greedy_bits(t, d)
    print(got, "OK" if got == exp else "SAI")

4 OK
7 OK
4 OK


**Kiểm tra nhanh** với `target = 3`, một lượt: `moves += 1 + 1 = 2`, `t = 3 >> 1 = 1` ⇒ đáp án 2. Đi ngược: 3 −1→ 2 ÷2→ 1 (2 bước); xuôi tương ứng: 1 ×2→ 2 +1→ 3 (2 bước) ✔.

Khi `t` không có lượt chia thì đúng là `t − 1` bước tăng. Khi `maxDoubles ≥ bit_length − 1`, vòng lặp chạy đúng `bit_length − 1` lần ⇒ đáp án = `(bit_length − 1) + (số bit 1 − 1)` (mỗi bit 1 trừ bit đầu cần một lần +1).

### 7.4 Cách 5 — Công thức đóng khi đủ lượt nhân đôi

In [13]:
def closed_form_when_enough(target, maxDoubles):
    L = target.bit_length()
    if maxDoubles >= L - 1:
        return (L - 1) + (bin(target).count("1") - 1)
    return greedy_bits(target, maxDoubles)

for t, d, exp in examples + [(10**9, 100, None), (2**29, 100, None)]:
    got = closed_form_when_enough(t, d)
    print(t, d, "->", got, "| khớp greedy:", got == greedy_iterative(t, d))

5 0 -> 4 | khớp greedy: True
19 2 -> 7 | khớp greedy: True
10 4 -> 4 | khớp greedy: True
1000000000 100 -> 41 | khớp greedy: True
536870912 100 -> 29 | khớp greedy: True


**Độ phức tạp cách 3/4/5:** O(min(maxDoubles, log target)) thời gian, O(1) bộ nhớ.

## 8. TRACE từng bước

### 8.1 Trace cách 3 trên `target = 100, maxDoubles = 3`

| vòng | t trước | lẻ? | hành động | t sau | còn lượt | moves |
|---|---|---|---|---|---|---|
| 1 | 100 | không | ÷2 | 50 | 2 | 1 |
| 2 | 50 | không | ÷2 | 25 | 1 | 2 |
| 3 | 25 | có | −1 | 24 | 1 | 3 |
| 4 | 24 | không | ÷2 | 12 | 0 | 4 |
| hết lượt | 12 | | +11 | 1 | | 15 |

In [14]:
def trace_greedy(target, maxDoubles):
    moves = 0; it = 0
    print(f"bắt đầu: t={target}, lượt={maxDoubles}")
    while target > 1 and maxDoubles > 0:
        it += 1; before = target
        if target % 2:
            target -= 1; act = "lẻ -> -1"
        else:
            target //= 2; maxDoubles -= 1; act = "chẵn -> /2"
        moves += 1
        print(f"vòng {it}: {before:>4} {act:<11} -> {target:<4} còn lượt {maxDoubles}, moves={moves}")
    print(f"cộng {target - 1} bước -1 -> đáp án {moves + target - 1}")
    return moves + target - 1

trace_greedy(100, 3)

bắt đầu: t=100, lượt=3
vòng 1:  100 chẵn -> /2  -> 50   còn lượt 2, moves=1
vòng 2:   50 chẵn -> /2  -> 25   còn lượt 1, moves=2
vòng 3:   25 lẻ -> -1    -> 24   còn lượt 1, moves=3
vòng 4:   24 chẵn -> /2  -> 12   còn lượt 0, moves=4
cộng 11 bước -1 -> đáp án 15


15

### 8.2 Dựng lại đường đi **xuôi** từ 1 (để thấy lời giải cụ thể)

In [15]:
def forward_path(target, maxDoubles):
    ops = []; t = target
    while t > 1 and maxDoubles > 0:
        if t % 2: ops.append("+1"); t -= 1
        else: ops.append("x2"); t //= 2; maxDoubles -= 1
    ops += ["+1"] * (t - 1)
    ops.reverse()                         # đảo lại thành thứ tự xuôi
    x = 1; seq = [1]
    for op in ops:
        x = x + 1 if op == "+1" else 2 * x; seq.append(x)
    return ops, seq

for t, d, _ in examples + [(100, 3, None)]:
    ops, seq = forward_path(t, d)
    print(f"target={t}, maxDoubles={d}: {len(ops)} bước, dãy giá trị {seq}")

target=5, maxDoubles=0: 4 bước, dãy giá trị [1, 2, 3, 4, 5]
target=19, maxDoubles=2: 7 bước, dãy giá trị [1, 2, 3, 4, 8, 9, 18, 19]
target=10, maxDoubles=4: 4 bước, dãy giá trị [1, 2, 4, 5, 10]
target=100, maxDoubles=3: 15 bước, dãy giá trị [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 24, 25, 50, 100]


### 8.3 Trace BFS nhỏ: số trạng thái đã thăm

In [16]:
for t, d in [(19, 2), (200, 5), (2000, 5)]:
    tm = time.perf_counter(); r = brute_bfs(t, d)
    print(f"target={t}, maxDoubles={d}: BFS = {r}, {(time.perf_counter()-tm)*1000:.1f} ms, greedy = {greedy_iterative(t, d)}")

target=19, maxDoubles=2: BFS = 7, 0.0 ms, greedy = 7
target=200, maxDoubles=5: BFS = 11, 0.1 ms, greedy = 11
target=2000, maxDoubles=5: BFS = 67, 1.6 ms, greedy = 67


## 9. Code nộp LeetCode

In [17]:
class Solution:
    def minMoves(self, target: int, maxDoubles: int) -> int:
        moves = 0
        while target > 1 and maxDoubles > 0:
            if target % 2 == 1:          # lẻ: bắt buộc trừ 1
                target -= 1
            else:                        # chẵn: chia đôi ngay
                target //= 2
                maxDoubles -= 1
            moves += 1
        return moves + (target - 1)      # hết lượt: chỉ còn trừ 1 (cộng thẳng, không lặp)

In [18]:
sol = Solution()
for t, d, exp in examples:
    got = sol.minMoves(t, d)
    print(got, "| kỳ vọng", exp, "| OK" if got == exp else "| SAI")
print("target lớn nhất:", sol.minMoves(10**9, 100), sol.minMoves(10**9, 0), sol.minMoves(10**9, 1))

4 | kỳ vọng 4 | OK
7 | kỳ vọng 7 | OK
4 | kỳ vọng 4 | OK
target lớn nhất: 41 999999999 500000000


## 10. Vì sao thuật toán đúng

Gọi `OPT(t, d)` là số bước ít nhất (đi ngược từ `t` về 1 với ≤ `d` lần chia đôi). Ta chứng minh tham lam đạt `OPT` bằng quy nạp theo `t`.

- `t = 1`: 0 bước. `d = 0`: chỉ trừ được ⇒ `t − 1`. ✔
- `t` lẻ, `t > 1`: không chia được số lẻ ⇒ bước đầu bắt buộc là −1 ⇒ `OPT(t, d) = 1 + OPT(t − 1, d)`, tham lam làm đúng vậy.
- `t` chẵn, `d ≥ 1`: xét lời giải tối ưu bất kỳ. Nếu nó chia ngay ⇒ trùng tham lam. Nếu không, nó trừ `m ≥ 1` lần rồi (a) chia ở `t − m` (m phải chẵn, `m = 2j`), hoặc (b) không bao giờ chia nữa.
  - (a): đi `t → t/2 − j` tốn `2j + 1`; tham lam đi `t → t/2 → t/2 − j` tốn `j + 1 ≤ 2j + 1`, dùng cùng số lượt chia ⇒ không tệ hơn.
  - (b): tốn `t − 1`; tham lam tốn `1 + OPT(t/2, d − 1) ≤ 1 + (t/2 − 1) = t/2 ≤ t − 1` (với `t ≥ 2`).
  ⇒ `OPT(t, d) = 1 + OPT(t/2, d − 1)`. ✔

Kết hợp quy nạp ⇒ tham lam tối ưu. Mục 12 còn kiểm chứng bằng BFS trên mọi `(target ≤ 200, maxDoubles ≤ 8)`. ∎

## 11. Phân tích Time / Space Complexity

`T = target ≤ 10⁹`, `D = maxDoubles ≤ 100`.

| Cách | Time | Space |
|---|---|---|
| 1. BFS | O(T·D) | O(T·D) |
| 1b. DP có nhớ | O(T·D) | O(T·D) + đệ quy sâu O(T) |
| 2. Tham lam đệ quy | O(min(D, log T)) | O(min(D, log T)) |
| 3. Tham lam lặp | O(min(D, log T)) | O(1) |
| 4/5. Bit / công thức | O(min(D, log T)) / O(log T) | O(1) |

Vì `log₂(10⁹) ≈ 30`, vòng lặp tham lam chạy tối đa ~60 lần dù `D = 100`.

In [19]:
t0 = time.perf_counter()
for _ in range(20000):
    Solution().minMoves(random.randint(1, 10**9), random.randint(0, 100))
print(f"20000 lần gọi với target ngẫu nhiên tới 1e9: {(time.perf_counter()-t0)*1000:.0f} ms")

20000 lần gọi với target ngẫu nhiên tới 1e9: 58 ms


## 12. Bộ test

### 12.1 Ví dụ đề + edge case

In [20]:
ALL = {"brute_bfs": brute_bfs, "dp_memo": dp_memo, "greedy_recursive": greedy_recursive,
       "greedy_iterative": greedy_iterative, "SolutionIter": SolutionIter().minMoves,
       "greedy_bits": greedy_bits, "closed_form": closed_form_when_enough, "Solution": Solution().minMoves}
edge = examples + [
    (1, 0, 0), (1, 100, 0), (2, 0, 1), (2, 1, 1), (3, 1, 2), (4, 1, 2), (4, 2, 2),
    (7, 100, 4),            # 1->2->3->6->7
    (8, 3, 3), (9, 3, 4),
    (1024, 100, 10), (1023, 100, 18),
]
for t, d, exp in edge:
    for name, f in ALL.items():
        if name == "dp_memo" and t > 300:
            continue                 # đệ quy sâu ~t: chỉ chạy dp_memo với target nhỏ
        assert f(t, d) == exp, (name, t, d, f(t, d), exp)
print("Edge case OK:", len(edge))

Edge case OK: 15


### 12.2 Kiểm chứng toàn bộ với BFS: mọi `target ≤ 200`, `maxDoubles ≤ 8`

In [21]:
cnt = 0
for t in range(1, 201):
    for d in range(0, 9):
        ref = brute_bfs(t, d)
        for name in ["greedy_recursive", "greedy_iterative", "SolutionIter", "greedy_bits", "closed_form", "Solution"]:
            assert ALL[name](t, d) == ref, (name, t, d)
        cnt += 1
print("đã kiểm", cnt, "cặp (target, maxDoubles): tất cả khớp BFS")

đã kiểm 1800 cặp (target, maxDoubles): tất cả khớp BFS


### 12.3 Input lớn: các cách tham lam khớp nhau; DP có nhớ khớp trên target vừa

In [22]:
rng = random.Random(2139)
for _ in range(3000):
    t = rng.randint(1, 10**9); d = rng.randint(0, 100)
    r = Solution().minMoves(t, d)
    assert r == greedy_recursive(t, d) == greedy_bits(t, d) == closed_form_when_enough(t, d)
for _ in range(200):
    t = rng.randint(1, 300); d = rng.randint(0, 10)
    assert dp_memo(t, d) == Solution().minMoves(t, d)
print("input lớn OK")

input lớn OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Tham lam **xuôi**: nhân đôi càng sớm càng tốt

In [23]:
def wrong_forward_double_early(target, maxDoubles):
    x, moves = 1, 0
    while x < target:
        if maxDoubles > 0 and 2 * x <= target:
            x *= 2; maxDoubles -= 1          # SAI: nhân đôi ngay khi có thể
        else:
            x += 1
        moves += 1
    return moves

print("bản sai :", wrong_forward_double_early(19, 2))
print("bản đúng:", Solution().minMoves(19, 2))

bản sai : 17
bản đúng: 7


**Vì sao sai:** nhân đôi `1 → 2 → 4` phí mất hai lượt quý giá ở lúc số còn nhỏ; sau đó phải +1 tới 19 (15 lần). Nhân đôi nên để dành cho lúc số **lớn**. **Sửa:** đi ngược và chia đôi khi chẵn.

### Lỗi 2 — Lặp từng bước −1 sau khi hết lượt (TLE với target lớn)

In [24]:
def slow_decrement(target, maxDoubles, cap=10**6):
    moves = 0; steps = 0
    while target > 1:
        if maxDoubles > 0 and target % 2 == 0:
            target //= 2; maxDoubles -= 1
        else:
            target -= 1                         # lặp 1e9 lần khi maxDoubles = 0!
        moves += 1; steps += 1
        if steps >= cap:
            return f"dừng sau {cap} vòng, mới tới target={target} (quá chậm)"
    return moves

print("bản chậm:", slow_decrement(10**9, 0))
print("bản đúng:", Solution().minMoves(10**9, 0))

bản chậm: dừng sau 1000000 vòng, mới tới target=999000000 (quá chậm)
bản đúng: 999999999


**Vì sao sai:** đúng kết quả nhưng tốn O(target) vòng = 10⁹ ⇒ TLE. **Sửa:** khi `maxDoubles == 0`, cộng thẳng `target − 1`.

### Lỗi 3 — Chia đôi số lẻ bằng `//` mà quên bước −1

In [25]:
def wrong_odd_half(target, maxDoubles):
    moves = 0
    while target > 1 and maxDoubles > 0:
        target //= 2; maxDoubles -= 1; moves += 1      # SAI: 19 // 2 = 9 nhưng 9*2 = 18 != 19
    return moves + target - 1

print("bản sai :", wrong_odd_half(19, 2))
print("bản đúng:", Solution().minMoves(19, 2))

bản sai : 5
bản đúng: 7


**Sửa:** số lẻ phải trừ 1 trước (tốn thêm 1 bước) — hoặc dùng `moves += 1 + (t & 1)` như cách 4.

### Lỗi 4 — Điều kiện dừng `target > 0` (đếm thừa)

In [26]:
def wrong_stop(target, maxDoubles):
    moves = 0
    while target > 0 and maxDoubles > 0:              # SAI: phải dừng ở 1
        if target % 2: target -= 1
        else: target //= 2; maxDoubles -= 1
        moves += 1
    return moves + max(target - 1, 0)

print("bản sai :", wrong_stop(10, 4))
print("bản đúng:", Solution().minMoves(10, 4))

bản sai : 5
bản đúng: 4


**Vì sao sai:** bắt đầu từ 1 nên mục tiêu ngược là **về 1**, không phải về 0. Bản sai đi `1 → 0` thêm một bước thừa. **Sửa:** `while target > 1 and maxDoubles > 0`.

### Lỗi 5 — Chia đôi cả khi `target = 2`… có sao không? (không sao — kiểm chứng)

Một nghi ngờ hợp lý: ở `t = 2`, chia đôi (1 bước) hay trừ 1 (1 bước) đều như nhau, nhưng chia đôi **tốn một lượt**. Vì sau đó `t = 1` nên lượt đó không còn giá trị — không ảnh hưởng đáp án. Đây không phải lỗi, nhưng hay làm người mới băn khoăn:

In [27]:
print([Solution().minMoves(2, d) for d in range(3)], [brute_bfs(2, d) for d in range(3)])

[1, 1, 1] [1, 1, 1]


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Các phép **tăng/nhân** từ số nhỏ tới đích ⇒ thử **đi ngược** từ đích (phép ngược thường có ít lựa chọn hơn: chỉ chia được khi chẵn).
- "Dùng tối đa k lần phép mạnh" ⇒ dùng phép mạnh ở chỗ **có lợi nhất** (ở đây: khi số lớn nhất — tức đầu tiên khi đi ngược).
- Ràng buộc `target ≤ 10⁹` ⇒ không được lặp O(target); phải cộng thẳng phần "chỉ còn +1".
- Kiểm chứng tham lam bằng BFS/DP cho mọi input nhỏ — thường chỉ mất vài giây.
- Bài cùng họ: 991 Broken Calculator (×2 và −1, đi ngược), 1342 Number of Steps to Reduce a Number to Zero, 397 Integer Replacement, 1404, 2571.

## 15. Biến thể của bài

### 15.1 Biến thể A — LC 991 Broken Calculator: từ `startValue`, phép `×2` và `−1` (không giới hạn), tới `target`

In [28]:
def broken_calc(startValue, target):
    moves = 0
    while target > startValue:
        target = target + 1 if target % 2 else target // 2
        moves += 1
    return moves + startValue - target

def broken_calc_bfs(s, t):
    lim = 2 * max(s, t) + 2
    dist = {s: 0}; q = deque([s])
    while q:
        x = q.popleft()
        if x == t: return dist[x]
        for y in (2 * x, x - 1):
            if 0 < y <= lim and y not in dist:
                dist[y] = dist[x] + 1; q.append(y)

print(broken_calc(2, 3), broken_calc(5, 8), broken_calc(3, 10))
for s in range(1, 25):
    for t in range(1, 60):
        assert broken_calc(s, t) == broken_calc_bfs(s, t)
print("khớp BFS")

2 2 3
khớp BFS


### 15.2 Biến thể B — Phép nhân **ba** (`x = 3x`, tối đa `k` lần) và +1

Đi ngược: nếu `t % 3 == 0` và còn lượt ⇒ chia 3; ngược lại trừ 1. Tham lam có còn đúng không? Kiểm chứng bằng BFS:

In [29]:
def triple_greedy(target, k):
    moves = 0
    while target > 1 and k > 0:
        if target % 3 == 0: target //= 3; k -= 1
        else: target -= 1
        moves += 1
    return moves + target - 1

def triple_bfs(target, k):
    dist = {(1, 0): 0}; q = deque([(1, 0)])
    while q:
        x, u = q.popleft()
        if x == target: return dist[(x, u)]
        for s in [(x + 1, u)] + ([(3 * x, u + 1)] if u < k else []):
            if s[0] <= target and s not in dist:
                dist[s] = dist[(x, u)] + 1; q.append(s)

bad = [(t, k) for t in range(1, 150) for k in range(0, 5) if triple_greedy(t, k) != triple_bfs(t, k)]
print("số phản ví dụ cho tham lam nhân 3:", len(bad), "| ví dụ:", bad[:5])

số phản ví dụ cho tham lam nhân 3: 0 | ví dụ: []


Tham lam "chia khi chia hết" vẫn đúng với ×3 trên dải kiểm thử (lập luận đổi chỗ tương tự: trừ `3j` rồi chia tốn `3j + 1`, chia rồi trừ `j` tốn `j + 1`). Nhưng nếu có **cả** ×2 và ×3 với giới hạn riêng thì tham lam đơn giản không còn chắc đúng — cần DP.

### 15.3 Biến thể C — Đếm **số đường đi tối ưu** khác nhau (với target nhỏ) bằng BFS đếm

In [30]:
def count_optimal_paths(target, maxDoubles):
    from collections import defaultdict
    dist = {(1, 0): 0}; ways = defaultdict(int); ways[(1, 0)] = 1
    q = deque([(1, 0)]); order = []
    while q:
        s = q.popleft(); order.append(s)
        x, k = s
        for t in [(x + 1, k)] + ([(2 * x, k + 1)] if k < maxDoubles else []):
            if t[0] > target: continue
            if t not in dist:
                dist[t] = dist[s] + 1; q.append(t)
            if dist[t] == dist[s] + 1:
                ways[t] += ways[s]
    best = min(dist[s] for s in dist if s[0] == target)
    return best, sum(ways[s] for s in dist if s[0] == target and dist[s] == best)

for t, d in [(19, 2), (10, 4), (8, 3), (2, 1)]:
    print((t, d), "-> (số bước, số đường tối ưu) =", count_optimal_paths(t, d))

(19, 2) -> (số bước, số đường tối ưu) = (7, 1)
(10, 4) -> (số bước, số đường tối ưu) = (4, 2)
(8, 3) -> (số bước, số đường tối ưu) = (3, 2)
(2, 1) -> (số bước, số đường tối ưu) = (1, 2)


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. BFS | đúng định nghĩa | O(T·D) | O(T·D) | kiểm chứng, T nhỏ |
| 1b. DP có nhớ | `f(t,d)` hai nhánh | O(T·D) | O(T·D) | khi chưa thấy tham lam |
| 2. Tham lam đệ quy | chẵn chia, lẻ trừ | O(min(D, log T)) | ngăn xếp | gọn, kiểu doocs |
| 3. Tham lam lặp | như 2, không đệ quy | O(min(D, log T)) | O(1) | **mặc định** |
| 4. Bit | `moves += 1 + (t & 1)`, `t >>= 1` | O(min(D, log T)) | O(1) | ngắn nhất |
| 5. Công thức | `(L−1) + (popcount−1)` khi đủ lượt | O(log T) | O(1) | hiểu sâu cấu trúc nhị phân |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `minMoves(656101987, 1)` và `minMoves(31, 100)`?

**Bài 2 (tính tay):** Trace tham lam ngược cho `target = 37, maxDoubles = 2` (ghi từng `t`).

**Bài 3 (điền TODO):** Viết `bt3_todo(target, maxDoubles)` trả về **danh sách phép xuôi** (`"+1"`/`"x2"`) của một lời giải tối ưu.

**Bài 4 (tự viết lại):** Viết `bt4_todo(target, maxDoubles)` cài cách 4 (bit) không nhìn code.

**Bài 5 (mini challenge):** Viết `bt5_todo(target)` trả về **số lần nhân đôi nhỏ nhất** `d` sao cho `minMoves(target, d)` đạt giá trị nhỏ nhất có thể (tức thêm lượt cũng không giảm được nữa).

In [31]:
def bt3_todo(target, maxDoubles):
    # TODO: danh sách phép xuôi tối ưu
    pass

def bt4_todo(target, maxDoubles):
    # TODO: cách bit
    pass

def bt5_todo(target):
    # TODO: số lượt nhân đôi tối thiểu để đạt số bước tối ưu tuyệt đối
    raise NotImplementedError

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** Mức 1: `maxDoubles = 1`: chia đôi đúng một lần (có thể sau một lần −1). Mức 2: 656101987 lẻ ⇒ −1 → 656101986, ÷2 → 328050993, rồi trừ thẳng. Mức 3: `1 + 1 + 328050992 = 328050994`; `31 = 11111₂` ⇒ `(5−1) + (5−1) = 8`.

**Bài 2.** Mức 1: 37 lẻ ⇒ −1. Mức 2: 37 → 36 (÷2, còn 1 lượt) → 18 (÷2, hết lượt) → 9. Mức 3: hết lượt ⇒ cộng `9 − 1 = 8`; tổng `3 + 8 = 11`.

**Bài 3.** Mức 1: ghi lại hành động khi đi ngược. Mức 2: "−1" ngược ↔ "+1" xuôi, "÷2" ↔ "x2". Mức 3: thêm `t − 1` lần "+1" ở cuối rồi **đảo** danh sách.

**Bài 4.** Mức 1: mỗi lượt dùng để chia. Mức 2: lẻ thì tốn thêm 1. Mức 3: `moves += 1 + (t & 1); t >>= 1; d -= 1`, cuối cộng `t − 1`.

**Bài 5.** Mức 1: với đủ lượt, tham lam chia `bit_length − 1` lần. Mức 2: nhưng ở `t = 2` thì chia hay trừ đều 1 bước. Mức 3: dự đoán đáp án = `max(0, bit_length − 2)`; để chắc chắn, cài bằng tìm kiếm tuyến tính trên `d` (tăng `d` tới khi `minMoves(target, d)` bằng giá trị tốt nhất) rồi so với công thức.

## 19. Đáp án tham khảo cho bài tập

**Bài 1:** `328050994`, `8`. **Bài 2:** `37 → 36 → 18 → 9`, rồi `+8` ⇒ `11`.

In [32]:
assert Solution().minMoves(656101987, 1) == 328050994
assert Solution().minMoves(31, 100) == 8
assert Solution().minMoves(37, 2) == 11

def bt3(target, maxDoubles):
    ops = []; t = target
    while t > 1 and maxDoubles > 0:
        if t % 2: ops.append("+1"); t -= 1
        else: ops.append("x2"); t //= 2; maxDoubles -= 1
    ops += ["+1"] * (t - 1)
    return ops[::-1]

def bt4(target, maxDoubles):
    moves = 0
    while maxDoubles and target > 1:
        moves += 1 + (target & 1); target >>= 1; maxDoubles -= 1
    return moves + target - 1

def bt5(target):
    best = Solution().minMoves(target, 100)
    d = 0
    while Solution().minMoves(target, d) != best:
        d += 1
    return d

for t in range(1, 300):
    for d in range(0, 10):
        ops = bt3(t, d)
        assert apply(ops) == t and len(ops) == Solution().minMoves(t, d) and ops.count("x2") <= d
        assert bt4(t, d) == Solution().minMoves(t, d)
for t in range(1, 2000):
    b = bt5(t)
    assert b == (max(0, t.bit_length() - 2) if t >= 2 else 0), (t, b)
print("bt5 khớp công thức max(0, bit_length - 2):", [bt5(t) for t in [1, 2, 3, 4, 19, 1024]])
print("Đáp án bài tập: tất cả assert OK")

bt5 khớp công thức max(0, bit_length - 2): [0, 0, 0, 1, 3, 9]
Đáp án bài tập: tất cả assert OK


Ghi chú bài 5: với `t ≥ 2`, tham lam đủ lượt chia `bit_length − 1` lần, nhưng lần chia cuối cùng là ở `t = 2` hoặc `t = 3 → 2 → 1` — tại đó "÷2" và "−1" tốn như nhau, nên lượt cuối là thừa ⇒ chỉ cần `bit_length − 2` lượt. Assert ở trên kiểm chứng công thức cho mọi `t < 2000`.

## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
moves = 0
while target > 1 and maxDoubles > 0:   # đi ngược
    if target & 1: target -= 1         # lẻ: -1
    else: target >>= 1; maxDoubles -= 1  # chẵn: /2 ngay
    moves += 1
return moves + target - 1              # cộng thẳng, KHÔNG lặp
```

**Checklist:**
- [ ] Đi **ngược** từ `target` về **1** (không phải về 0).
- [ ] Số lẻ: trừ 1 trước khi chia.
- [ ] Hết lượt chia ⇒ cộng `target − 1` một lần (tránh O(10⁹)).
- [ ] `target = 1` ⇒ 0; `maxDoubles = 0` ⇒ `target − 1`.
- [ ] Không nhân đôi sớm khi đi xuôi.